In [1]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.colors import HexColor, white
from reportlab.lib.units import mm
from reportlab.pdfgen import canvas
import pandas as pd
import json, os, datetime
pd.set_option('display.max_columns', None)

In [2]:
RESULTS = 'results'
OUTPUT  = 'release/certificate.pdf'
LOGO    = 'images/measure.png'
SIGN    = 'images/signature.png'

OPTIONS  = json.loads(open('../info.json', 'r').read())
TEST     = OPTIONS.get('test', 1)
RECORD   = f'../Dataset/test{TEST}/results'
strategy = json.loads(open(f'{RECORD}/model.json', 'r').read())
filters  = json.loads(open(f'{RECORD}/filters.json', 'r').read())

INFO = {
    'Modelo do MRU':      'Measure MRU',
    'Número de Série':    OPTIONS.get('id', '0001'),
    'Nº do Certificado':  f'MO-{datetime.date.today():%Y%m%d}-{TEST:04d}',
    'Data da Calibração': datetime.date.today().strftime('%d/%m/%Y'),
    'Estratégia':         f'{strategy["strategy"]} ({filters["strategy"]})',
    'Técnico':            'Measure Offshore'
}

INFO

{'Modelo do MRU': 'Measure MRU',
 'Número de Série': 'MIC926199',
 'Nº do Certificado': 'MO-20260922-0001',
 'Data da Calibração': '22/09/2026',
 'Estratégia': 'kalman_fusion (filters)',
 'Técnico': 'Measure Offshore'}

# REQUISITOS
- Cada seção cobra um limite de cada métrica sobre um grupo de eixos, e o status é sempre o pior eixo contra o limite
- Roll e pitch ganham uma coluna cada (`detail`); giroscópio e acelerômetro entram numa coluna só, com o pior dos três
- `gap` é o respiro antes do título da seção e `widths` a divisão das colunas: são as medidas da folha, não da medição

In [3]:
PLOTTED = ['roll', 'pitch']    # OS DOIS ÂNGULOS QUE O PDF DESENHA

SECTIONS = [
    {
        'title':  'TESTES DE ACURÁCIA DE ROLL & PITCH',
        'about':  'O MRU foi submetido a movimentos dinâmicos conhecidos. Comparação entre medido e referência.',
        'axes':   PLOTTED,
        'tests':  [('RMS estático (deg)', 'rms_stat', 0.20), ('RMS dinâmico (deg)', 'rms_dyn', 0.50), ('Scale Factor', 'rms_scale_factor', 0.10)],
        'detail': True,
        'plots':  True,
        'gap':    5,
        'widths': [0.30, 0.18, 0.16, 0.16, 0.20]
    },
    {
        'title':  'TESTES DE GIROSCÓPIO',
        'about':  'Verificação através de rotações contínuas em velocidades conhecidas.',
        'axes':   ['wx', 'wy', 'wz'],
        'tests':  [('Ruído Estático STD (deg/s)', 'std_stat', 0.30)],
        'detail': False,
        'plots':  False,
        'gap':    5,
        'widths': [0.40, 0.20, 0.20, 0.20]
    },
    {
        'title':  'TESTE DE ACELERÔMETRO',
        'about':  'Verificação inclinando o MRU em degraus de 90° em um círculo completo.',
        'axes':   ['ax', 'ay', 'az'],
        'tests':  [('Ruído Estático STD (m/s²)', 'std_stat', 0.05)],
        'detail': False,
        'plots':  False,
        'gap':    15,
        'widths': [0.35, 0.22, 0.22, 0.21]
    }
]

pd.DataFrame(SECTIONS)[['title', 'axes', 'detail', 'plots']]

,title,axes,detail,plots
0,TESTES DE ACURÁCIA DE ROLL & PITCH,"[roll, pitch]",True,True
1,TESTES DE GIROSCÓPIO,"[wx, wy, wz]",False,False
2,TESTE DE ACELERÔMETRO,"[ax, ay, az]",False,False


# DADOS DO ENSAIO
- O `Analysis.ipynb` deixa em `results/` um `metrics.json` por eixo e, para cada ângulo desenhado, os dois png no tamanho da folha
- Sem os gráficos de roll e de pitch o certificado não sai: eles são o corpo da primeira seção

In [4]:
# O QUE O Analysis.ipynb DEIXOU EM results/
class ReportData:
    def __init__(self, folder=RESULTS):
        self.folder = folder

    def update(self):
        self.metrics = {}
        self.plots   = {}

        for axis in sorted(os.listdir(self.folder)):
            self.metrics[axis] = json.loads(open(f'{self.folder}/{axis}/metrics.json', 'r').read())

            for name in ['ref_vs_model', 'error']:
                path = f'{self.folder}/{axis}/{name}.png'

                if os.path.exists(path):
                    self.plots[f'{axis}_{name}'] = path

        missing = [axis for axis in PLOTTED if f'{axis}_error' not in self.plots]

        if missing:
            raise FileNotFoundError(f'faltam os gráficos de {missing} em {self.folder}: rode o Analysis.ipynb')

    def status(self, value, limit):
        return 'Aprovado' if value <= limit else 'Reprovado'

    def getHeaders(self, section):
        measured = [axis.capitalize() for axis in section['axes']] if section['detail'] else ['Medido']
        return ['Parâmetro', 'Requisito'] + measured + ['Status']

    # UMA LINHA POR REQUISITO, COM O VALOR DE CADA EIXO OU SÓ O PIOR DELES; EIXO QUE NÃO MEDIU AQUILO FICA DE FORA
    def getRows(self, section):
        rows = []

        for label, key, limit in section['tests']:
            values = [self.metrics.get(axis, {}).get(key) for axis in section['axes']]
            values = [value for value in values if value is not None]

            if not values:
                continue

            cells = values if section['detail'] else [max(values)]
            rows.append([label, f'≤ {limit:.4f}'] + [f'{value:.4f}' for value in cells] + [self.status(max(values), limit)])

        return rows

    def info(self):
        return self.metrics


data = ReportData()
data.update()

pd.DataFrame(data.info()).T

,std_stat,std_raw,r2,mae,rmse,precision,rms_stat,rms_dyn,rms_scale_factor,time_drift
ax,0.006697,0.022026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ay,0.013108,0.024123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
az,0.013254,0.019704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pitch,0.124731,NaN,-278.865477,8.099771,8.431371,0.225901,7.752072,9.210306,13.072308,-3.100342
roll,0.098158,NaN,0.996620,0.303600,0.451870,0.197236,0.183824,0.645896,0.011468,0.272209
wx,0.222940,0.223651,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wy,0.129707,0.203903,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wz,0.158389,0.261326,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
yaw,4.145420,NaN,-25.771895,5.887483,6.896866,7.899227,5.577160,8.254957,5.426598,-253.226548


# TABELAS
- O mesmo conteúdo que vai para a folha, antes de virar PDF

In [5]:
for number, section in enumerate(SECTIONS, start=1):
    print(f'{number}. {section["title"]}')
    display(pd.DataFrame(data.getRows(section), columns=data.getHeaders(section)))

1. TESTES DE ACURÁCIA DE ROLL & PITCH


,Parâmetro,Requisito,Roll,Pitch,Status
0,RMS estático (deg),≤ 0.2000,0.1838,7.7521,Reprovado
1,RMS dinâmico (deg),≤ 0.5000,0.6459,9.2103,Reprovado
2,Scale Factor,≤ 0.1000,0.0115,13.0723,Reprovado


2. TESTES DE GIROSCÓPIO


,Parâmetro,Requisito,Medido,Status
0,Ruído Estático STD (deg/s),≤ 0.3000,0.2229,Aprovado


3. TESTE DE ACELERÔMETRO


,Parâmetro,Requisito,Medido,Status
0,Ruído Estático STD (m/s²),≤ 0.0500,0.0133,Aprovado


# DESENHO DA FOLHA
- Uma folha A4: cabeçalho, os dados do ensaio, as seções de requisitos com a tabela de cada uma, os gráficos de roll e pitch e o rodapé
- Todo `draw` recebe o `y` onde começa e devolve o `y` onde parou, que é como a página desce

In [6]:
DARK   = HexColor('#3A1078')
GREEN  = HexColor('#2E7D32')
CHECK  = HexColor('#4CAF50')
TEXT   = HexColor('#333333')
GRAY   = HexColor('#E0E0E0')
HEADER = HexColor('#6A3CBC')
STRIPE = HexColor('#F3EDFA')


# A FOLHA DO CERTIFICADO, DE CIMA PARA BAIXO
class PDFReport:
    W, H   = A4
    MARGIN = 15 * mm

    def __init__(self, data, output=OUTPUT):
        self.data   = data
        self.output = output

        os.makedirs(os.path.dirname(output), exist_ok=True)
        self.canvas = canvas.Canvas(output, pagesize=A4)

    def width(self):
        return self.W - 2*self.MARGIN

    def drawHeader(self):
        c = self.canvas
        height = 55

        c.setFillColor(DARK)
        c.rect(0, self.H - height, self.W, height, fill=1, stroke=0)

        if os.path.exists(LOGO):
            c.drawImage(LOGO, 12, self.H - height + 8, width=120, height=40, preserveAspectRatio=True, mask='auto')

        c.setFillColor(white)
        c.setFont('Helvetica-Bold', 18); c.drawRightString(self.W - 15, self.H - 28, 'CERTIFICADO DE CALIBRAÇÃO')
        c.setFont('Helvetica', 12);      c.drawRightString(self.W - 15, self.H - 44, 'MRU')

    # OS DADOS À ESQUERDA E, À DIREITA, O TEXTO QUEBRADO NA LARGURA QUE SOBRA MAIS A ASSINATURA
    def drawInfo(self, top):
        c = self.canvas
        y = top - 10

        for label, value in INFO.items():
            c.setFillColor(TEXT)
            c.setFont('Helvetica-Bold', 7.5); c.drawString(self.MARGIN + 5, y, f'{label}:')
            c.setFont('Helvetica', 7.5);      c.drawString(self.MARGIN + 125, y, str(value))
            y -= 13

        left  = self.W/2 + 20
        limit = self.W - self.MARGIN - left - 10
        block = c.beginText(left, top - 10)
        block.setFont('Helvetica', 7)
        block.setFillColor(TEXT)

        line = ''
        for word in 'Este certificado atesta que o sistema MRU foi calibrado conforme procedimentos internos.'.split():
            if c.stringWidth(f'{line} {word}'.strip(), 'Helvetica', 7) < limit:
                line = f'{line} {word}'.strip()
            else:
                block.textLine(line)
                line = word

        block.textLine(line)
        c.drawText(block)

        if os.path.exists(SIGN):
            c.drawImage(SIGN, left + 60, top - 70, width=90, height=30, preserveAspectRatio=True, mask='auto')

        c.setFont('Helvetica', 6.5)
        c.drawString(left + 60, top - 78, 'Eng. de Calibração')
        return top - 100

    def drawSection(self, top, number, section):
        c = self.canvas

        c.setFont('Helvetica-Bold', 9); c.setFillColor(DARK)
        c.drawString(self.MARGIN, top, f'{number}. {section["title"]}')

        c.setFont('Helvetica', 6.5); c.setFillColor(TEXT)
        c.drawString(self.MARGIN, top - 14, section['about'])
        return top - 26

    def drawTable(self, top, headers, rows, widths):
        c      = self.canvas
        table  = self.width()
        widths = [table*fraction for fraction in widths]
        x, y   = self.MARGIN, top

        c.setFillColor(HEADER)
        c.rect(x, y - 18, table, 18, fill=1, stroke=0)
        c.setFillColor(white)
        c.setFont('Helvetica-Bold', 7)

        cx = x
        for i, header in enumerate(headers):
            c.drawCentredString(cx + widths[i]/2, y - 12, header)
            cx += widths[i]

        y -= 18

        for line, row in enumerate(rows):
            if line % 2 == 1:
                c.setFillColor(STRIPE)
                c.rect(x, y - 16, table, 16, fill=1, stroke=0)

            cx = x
            for i, cell in enumerate(row):
                approved = (str(cell) == 'Aprovado')
                c.setFillColor(GREEN if approved else TEXT)
                c.setFont('Helvetica-Bold' if approved else 'Helvetica', 6.5)
                c.drawCentredString(cx + widths[i]/2, y - 11, str(cell))
                cx += widths[i]

            y -= 16

        c.setStrokeColor(GRAY)
        c.setLineWidth(0.3)
        c.line(x, y, x + table, y)
        return y - 5

    # REFERÊNCIA x MEDIDO EM CIMA, ERRO EMBAIXO, UM ÂNGULO POR COLUNA
    def drawPlots(self, top):
        gap    = 8
        width  = (self.width() - gap) / 2
        height = width * (150/250)
        y      = top

        for name in ['ref_vs_model', 'error']:
            for column, axis in enumerate(PLOTTED):
                self.canvas.drawImage(self.data.plots[f'{axis}_{name}'], self.MARGIN + column*(width + gap), y - height, width=width, height=height, mask='auto')

            y -= height + 5

        return y

    def drawFooter(self):
        c = self.canvas
        x, y = self.MARGIN + 12, 17.5

        c.setFillColor(HexColor('#F0F7F0'))
        c.rect(0, 0, self.W, 35, fill=1, stroke=0)
        c.setStrokeColor(CHECK)
        c.setLineWidth(1)
        c.line(0, 35, self.W, 35)

        c.setFillColor(CHECK)
        c.circle(x, y, 10, fill=1, stroke=0)
        c.setFillColor(white)
        c.setFont('Helvetica-Bold', 14); c.drawCentredString(x, y - 5, '✓')

        c.setFillColor(GREEN)
        c.setFont('Helvetica-Bold', 6.5); c.drawString(x + 18, y + 4, 'O sistema MRU atende a todos os requisitos de calibração.')
        c.setFillColor(TEXT)
        c.setFont('Helvetica', 5.5);      c.drawString(x + 18, y - 6, 'Recomendamos nova calibração em 12 meses.')

        c.setFillColor(DARK)
        c.setFont('Helvetica-Bold', 7);   c.drawRightString(self.W - self.MARGIN, y + 5, 'measureoffshore.com')

    def export(self):
        self.drawHeader()
        y = self.drawInfo(self.H - 65)

        for number, section in enumerate(SECTIONS, start=1):
            y = self.drawSection(y - section['gap'], number, section)
            y = self.drawTable(y - 3, self.data.getHeaders(section), self.data.getRows(section), section['widths'])

            if section['plots']:
                y = self.drawPlots(y - 3)

        self.drawFooter()
        self.canvas.save()
        return self.output

# GERANDO O CERTIFICADO

In [7]:
report = PDFReport(data)
print(report.export())

release/certificate.pdf
